# MDE by SMN — MiniMax H3 Colab Gateway

Zero-budget gateway for the Android app. **The T4 is used to host the gateway only; MiniMax H3 inference remains on the Hugging Face Space because the current unquantized H3 model is far larger than a 16 GB T4.**

The gateway calls `observantdistressed/minimax-h3` through `gradio_client`, and exposes a Gradio URL that the Android app can open. Hugging Face documents public Gradio Spaces as callable APIs.

In [ ]:
!pip -q install -U gradio gradio_client
print('Gateway dependencies installed.')

In [ ]:
import os, time
from datetime import datetime
from zoneinfo import ZoneInfo
import gradio as gr
from gradio_client import Client

SPACE = 'observantdistressed/minimax-h3'
DAILY_LIMIT = 5  # Change this if you want a different local app limit.
TZ = ZoneInfo('Asia/Kolkata')
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()

client = Client(SPACE, hf_token=HF_TOKEN or None)
print('Connected to:', SPACE)
client.view_api()

In [ ]:
usage = {'date': None, 'used': 0}

def usage_state():
    today = datetime.now(TZ).date().isoformat()
    if usage['date'] != today:
        usage['date'] = today
        usage['used'] = 0
    remaining = max(0, DAILY_LIMIT - usage['used'])
    now = datetime.now(TZ)
    next_reset = (now.replace(hour=0, minute=0, second=0, microsecond=0).timestamp() + 86400)
    seconds = max(0, int(next_reset - now.timestamp()))
    h, rem = divmod(seconds, 3600)
    m = rem // 60
    return f'**Usage:** {usage["used"]} / {DAILY_LIMIT} used  ·  **{remaining} remaining**  ·  **reset in {h}h {m}m (IST)**'

def generate(prompt, image, last_image, canvas, duration, steps, seed):
    usage_state()
    if usage['used'] >= DAILY_LIMIT:
        raise gr.Error('Daily limit reached. The quota resets at midnight IST.')
    if not prompt or not prompt.strip():
        raise gr.Error('Enter a prompt first.')
    started = time.time()
    result = client.predict(
        prompt,
        image,
        last_image,
        canvas,
        int(duration),
        int(steps),
        int(seed),
        False,  # upsample
        False,  # turbo
        (),     # nsfw_lora_keys — deliberately empty
        '',     # custom_lora_url
        1.0,    # custom_lora_scale
        '',     # custom_lora_trigger
        api_name='/generate'
    )
    usage['used'] += 1
    report = result[1] if isinstance(result, (tuple, list)) and len(result) > 1 else ''
    elapsed = int(time.time() - started)
    return result[0], f'{report}\n\nGateway time: {elapsed}s', usage_state()

def status():
    return usage_state()

CANVASES = [
    '16:9 (768x1344)', '9:16 (1344x768)', '1:1 (768x768)',
    '4:3 (768x1024)', '3:4 (1024x768)', '21:9 (672x1536)'
]

with gr.Blocks(title='MDE by SMN × MiniMax H3') as demo:
    gr.Markdown('# MDE by SMN × MiniMax H3\nGenerate video + soundtrack through the Hugging Face Space.')
    usage_box = gr.Markdown(usage_state())
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label='Prompt', lines=4, placeholder='Describe the video you want to create...')
            with gr.Row():
                image = gr.Image(label='First keyframe (optional)', type='filepath')
                last_image = gr.Image(label='Last keyframe (optional)', type='filepath')
            canvas = gr.Dropdown(CANVASES, value=CANVASES[0], label='Canvas')
            duration = gr.Slider(5, 14, value=5, step=1, label='Duration (seconds)')
            steps = gr.Slider(10, 40, value=30, step=1, label='Steps')
            seed = gr.Number(value=42, precision=0, label='Seed')
            run = gr.Button('Generate', variant='primary')
        with gr.Column():
            video = gr.Video(label='Video + soundtrack')
            report = gr.Markdown()
    run.click(generate, [prompt, image, last_image, canvas, duration, steps, seed], [video, report, usage_box], api_name='generate_gateway')
    demo.load(status, None, usage_box)

print('Launching the public Colab gateway...')
demo.queue(max_size=2).launch(share=True, inline=False, show_error=True)

## Android connection
Copy the `https://....gradio.live` URL printed by the final cell and paste it into the MDE H3 Android app. The URL normally changes when the Colab runtime is restarted.

## Optional Hugging Face token
For a public Space, authentication is not necessarily required. If you have an authorized Hugging Face token, you can store it in Colab Secrets as `HF_TOKEN` and restart the runtime. Do **not** hard-code the token into the APK.